Action Items:
- Pick a column of data from FREDMD
- Decide train, validation, test split
- determine evaluation procedure
  - doesn't need to be optimized im pretty sure, just have reasoning


**Forecasting Setup**

Train on 6 years of data -> evaluate on next six years -> repeat
- justification -> average business cycle

Train the model on six years of historical data, then evaluate its forecasts on the following six years. Repeat this process by moving the train and evaluation windows forward through time.

The six-year window is chosen to roughly match the length of an average business cycle, so each evaluation period is more likely to include different economic conditions, such as expansion and contraction. This helps measure whether the model performs consistently across changing market regimes rather than only during one short period.

Forecasting Baseline: y_hat<sub>t+1</sub> = y<sub>t</sub>

Train for 6 years of data at a time with a one year lag





**Implementation note:** The six-year rolling-window proposal above is an earlier design idea. The implementation below uses the chronological 70% training / 15% validation / 15% test split in our agreed experimental-design table. The neural models are independent of this split, so a rolling-window extension can reuse them later.

In [1]:
from pathlib import Path
import math
from typing import NamedTuple

import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader, TensorDataset

# Colab: upload the provided CSV as data.csv, or keep its original filename.
DATA_PATH = Path("data.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("2026-rev-08-MD.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Upload data.csv or 2026-rev-08-MD.csv before running this cell.")

columns = ["S&P 500", "FEDFUNDS", "UNRATE"]
df = pd.read_csv(DATA_PATH)
# FRED-MD's Transform: row is metadata, not a monthly observation.
df = df.loc[df["sasdate"].astype(str).str.strip().ne("Transform:")].copy()
extracted = df[["sasdate", *columns]].copy()
extracted["sasdate"] = pd.to_datetime(extracted["sasdate"], format="%m/%d/%Y")
extracted[columns] = extracted[columns].apply(pd.to_numeric, errors="raise")
extracted = extracted.sort_values("sasdate").reset_index(drop=True)

months = pd.PeriodIndex(extracted["sasdate"], freq="M")
expected_months = pd.period_range(months[0], periods=len(months), freq="M")
if not months.equals(expected_months):
    raise ValueError("The data must contain one row per consecutive month.")

# Keep missing-value rows to preserve the calendar. Window creation below
# excludes examples whose inputs or target contain non-finite values.
# Values are used as supplied; no differencing or log transform is applied.
extracted.to_csv("selected_columns.csv", index=False)
print(extracted.head())
print("Missing values by series:")
print(extracted[columns].isna().sum())

     sasdate  S&P 500  FEDFUNDS  UNRATE
0 1959-01-01    55.62      2.48     6.0
1 1959-02-01    54.77      2.43     5.9
2 1959-03-01    56.16      2.80     5.6
3 1959-04-01    57.10      2.96     5.2
4 1959-05-01    57.96      2.90     5.1
Missing values by series:
S&P 500     0
FEDFUNDS    0
UNRATE      1
dtype: int64


3. Reversible Instance Normalization
Goal is to use the window specific data, instead of considering future values when calculating normalization statistics.

The paper normalises each input window using statistics
from that window only, and it lets the network forecast on the normalised scale, and then undoes the normalisation before the prediction is compared with the target

The cells below implement RevIN directly in PyTorch so gradients pass through the forecast. We use `scale = sqrt(variance + 1e-5)` and reverse normalization with the same scale. Statistics come only from the historical input window. This uses the non-affine variant without optional learned scale/offset parameters. The models return predictions on the supplied series' scale, so do not normalize input arrays again or manually denormalize their predictions.

**Data preparation:** Keep the monthly date rows intact and skip windows with a missing input or target. The CSV contains a `Transform:` metadata row and appears to contain levels, whereas the assignment describes already-transformed data. This notebook uses numeric values as supplied; confirm the intended data preparation before interpreting economic results.

In [2]:
p = 12
MODEL_CONFIG = dict(lags=p, hidden_dim=32, num_layers=1, dropout=0.0)
TRAINING_CONFIG = dict(learning_rate=0.001, batch_size=32,
                       max_epochs=200, patience=20)
SEEDS = (1, 2, 3, 4, 5)

| Choice | What it means | Suggested setting and basic justification |
|---|---|---|
| Number of lags | How many past months the model sees. | **12 months.** One year provides recent history while keeping the input small. |
| Train/validation/test split | Which data train the model, guide training decisions, and measure final performance. | **First 70% / next 15% / final 15%, chronologically.** This gives the model plenty of training data and evaluates its ability to predict later observations. Use common date boundaries across all three series. |
| Hidden dimension | How many internal features the model can learn. | **32 units.** This allows the model to learn useful patterns while keeping it small for a monthly dataset. |
| Number of layers | How many processing layers the model uses. | **One hidden layer for the fully connected model; one LSTM layer for both LSTM models.** Simple architectures make the models easier to train and compare. |
| Activation function | The function that allows the model to learn nonlinear relationships. | **ReLU for the fully connected hidden layer; standard sigmoid/tanh inside the LSTM.** Use a linear output layer so forecasts can take any numerical value. |
| Optimizer | The method used to update the model’s weights. | **Adam.** It adjusts updates separately for different weights and is a straightforward starting choice. |
| Learning rate | How large the weight updates are during training. | **0.001, held constant.** This is Adam’s default learning rate in PyTorch and provides a reasonable starting point. |
| Batch size | How many training examples are processed before each weight update. | **32 input windows.** This provides several examples per update while allowing multiple updates per training epoch. |
| Dropout | Randomly disabling some internal units during training to discourage overfitting. | **0: no dropout.** Small networks and early stopping keep the design simple while helping limit overfitting. |
| Stopping criterion | When to stop training. | **Maximum of 200 epochs, stopping after 20 epochs without improved validation MSE.** Restore the weights from the best validation epoch to avoid keeping a model that has begun to overfit. |
| Evaluation metrics | How forecast accuracy is measured. | **RMSE and MAE.** RMSE emphasizes large mistakes, while MAE measures average absolute error and is less sensitive to extreme errors. Lower values are better. |

In [3]:
def make_windows(series, p):
    """Return finite (past p months, next month) examples, in time order."""
    series = np.asarray(series, dtype=np.float32)
    if series.ndim != 1 or not isinstance(p, int) or p < 1:
        raise ValueError("Use a one-dimensional series and a positive integer p.")
    X, y = [], []
    for i in range(p, len(series)):
        window = series[i-p:i]
        target = series[i]
        if np.isfinite(window).all() and np.isfinite(target):
            X.append(window)
            y.append(target)
    return np.asarray(X, dtype=np.float32).reshape(-1, p), np.asarray(y, dtype=np.float32)

In [4]:
def prepare_datasets(df, column_name, p):
    """Split by target month, keeping the original calendar boundaries.

    Validation/test inputs can use earlier observed history. Invalid windows
    are excluded after boundaries are fixed; no future-value imputation is used.
    All models and the baseline for a column use the same retained examples.
    """
    series = df[column_name].to_numpy(dtype=np.float32)
    if not isinstance(p, int) or p < 1 or len(series) <= p:
        raise ValueError("p must be positive and smaller than the series length.")
    train_end = int(0.7 * len(series))
    val_end = train_end + int(0.15 * len(series))
    splits = {name: ([], []) for name in ("train", "val", "test")}

    for i in range(p, len(series)):
        window, target = series[i-p:i], series[i]
        if not (np.isfinite(window).all() and np.isfinite(target)):
            continue
        name = "train" if i < train_end else "val" if i < val_end else "test"
        splits[name][0].append(window)
        splits[name][1].append(target)

    result = []
    for name in ("train", "val", "test"):
        X, y = splits[name]
        if not X:
            raise ValueError(f"No usable {name} examples for {column_name}.")
        result.extend((np.asarray(X, dtype=np.float32).reshape(-1, p),
                       np.asarray(y, dtype=np.float32)))
    return tuple(result)

In [5]:
class WindowStatistics(NamedTuple):
    mean: Tensor
    scale: Tensor


class RevIN(nn.Module):
    """Window normalization and its inverse, without learnable affine terms.

    Statistics are per example, across time only. Passing them explicitly
    prevents one forward call from overwriting another window's statistics.
    This is the simple non-affine RevIN variant; the homework does not require
    the optional learnable affine transformation.
    """

    def __init__(self, eps: float = 1e-5):
        super().__init__()
        if not math.isfinite(eps) or eps <= 0:
            raise ValueError("eps must be finite and positive.")
        self.eps = eps

    def normalize(self, x: Tensor) -> tuple[Tensor, WindowStatistics]:
        if x.ndim != 3 or x.shape[1] == 0 or x.shape[2] != 1:
            raise ValueError("Expected input shape (batch, lags, 1).")
        if not x.is_floating_point():
            raise TypeError("Inputs must be floating-point tensors.")
        mean = x.mean(dim=1, keepdim=True).detach()
        scale = (x.var(dim=1, keepdim=True, unbiased=False) + self.eps).sqrt().detach()
        return (x - mean) / scale, WindowStatistics(mean, scale)

    def denormalize(self, x: Tensor, stats: WindowStatistics) -> Tensor:
        """Restore a normalized window or forecast of shape (batch, time, 1)."""
        return x * stats.scale + stats.mean

In [6]:
class _Forecaster(nn.Module):
    def __init__(self, lags: int, hidden_dim: int, num_layers: int,
                 dropout: float, eps: float):
        super().__init__()
        for name, value in (("lags", lags), ("hidden_dim", hidden_dim),
                            ("num_layers", num_layers)):
            if isinstance(value, bool) or not isinstance(value, int) or value < 1:
                raise ValueError(f"{name} must be a positive integer.")
        if not 0 <= dropout < 1:
            raise ValueError("dropout must be in [0, 1).")
        self.lags = lags
        self.revin = RevIN(eps)

    def _normalize(self, x: Tensor) -> tuple[Tensor, WindowStatistics]:
        if x.ndim != 3 or x.shape[1:] != (self.lags, 1):
            raise ValueError(f"Expected input shape (batch, {self.lags}, 1).")
        return self.revin.normalize(x)

    def _restore(self, prediction: Tensor, stats: WindowStatistics) -> Tensor:
        # Keep the batch dimension even when the batch contains one example.
        return self.revin.denormalize(prediction.unsqueeze(1), stats).squeeze(1)

In [7]:
def persistence_predict(X):
    return X[:, -1]

In [8]:
def _metric_arrays(y_true, y_pred):
    actual = np.asarray(y_true).reshape(-1)
    predicted = np.asarray(y_pred).reshape(-1)
    if actual.size == 0 or actual.shape != predicted.shape:
        raise ValueError("Actual values and predictions must have equal, nonzero lengths.")
    return actual, predicted


def mae(y_true, y_pred):
    actual, predicted = _metric_arrays(y_true, y_pred)
    return np.mean(np.abs(actual - predicted))


def rmse(y_true, y_pred):
    actual, predicted = _metric_arrays(y_true, y_pred)
    return np.sqrt(np.mean((actual - predicted) ** 2))

In [9]:
columns = ["S&P 500", "FEDFUNDS", "UNRATE"]

for column_name in columns:
    X_train, y_train, X_val, y_val, X_test, y_test = prepare_datasets(
        extracted,
        column_name,
        p
    )

    print("\n==============================")
    print(column_name)
    print("==============================")

    print("TRAIN")
    print("X_train shape:", X_train.shape)
    print("y_train shape:", y_train.shape)
    print("First X_train:", X_train[0])
    print("First y_train:", y_train[0])

    print("\nVALIDATION")
    print("X_val shape:", X_val.shape)
    print("y_val shape:", y_val.shape)
    print("First X_val:", X_val[0])
    print("First y_val:", y_val[0])

    print("\nTEST")
    print("X_test shape:", X_test.shape)
    print("y_test shape:", y_test.shape)
    print("First X_test:", X_test[0])
    print("First y_test:", y_test[0])


S&P 500
TRAIN
X_train shape: (555, 12)
y_train shape: (555,)
First X_train: [55.62 54.77 56.16 57.1  57.96 57.46 59.74 59.4  57.05 57.   57.23 59.06]
First y_train: 58.03

VALIDATION
X_val shape: (121, 12)
y_val shape: (121,)
First X_val: [1164.43 1178.28 1202.25 1222.24 1224.27 1225.92 1191.96 1237.37 1262.07
 1278.73 1276.65 1293.74]
First y_val: 1302.17

TEST
X_test shape: (123, 12)
y_test shape: (123,)
First X_test: [2111.94 2099.29 2094.14 2039.87 1945.41 2071.18 2080.62 2054.08 1918.6
 1903.03 2021.95 2075.54]
First y_test: 2065.55

FEDFUNDS
TRAIN
X_train shape: (555, 12)
y_train shape: (555,)
First X_train: [2.48 2.43 2.8  2.96 2.9  3.39 3.47 3.5  3.76 3.98 4.   3.99]
First y_train: 3.99

VALIDATION
X_val shape: (121, 12)
y_val shape: (121,)
First X_val: [2.79 3.   3.04 3.26 3.5  3.62 3.78 4.   4.16 4.29 4.49 4.59]
First y_val: 4.79

TEST
X_test shape: (123, 12)
y_test shape: (123,)
First X_test: [0.12 0.13 0.13 0.14 0.14 0.12 0.12 0.24 0.34 0.38 0.36 0.37]
First y_test: 0.37



In [10]:

columns = ["S&P 500", "FEDFUNDS", "UNRATE"]

for column_name in columns:
    X_train, y_train, X_val, y_val, X_test, y_test = prepare_datasets(
        extracted,
        column_name,
        p
    )

    # Persistence predictions
    val_predictions = persistence_predict(X_val)
    test_predictions = persistence_predict(X_test)

    print("\n==============================")
    print(column_name)
    print("==============================")

    print("Validation MAE:", mae(y_val, val_predictions))
    print("Validation RMSE:", rmse(y_val, val_predictions))

    print("Test MAE:", mae(y_test, test_predictions))
    print("Test RMSE:", rmse(y_test, test_predictions))


S&P 500
Validation MAE: 39.744293
Validation RMSE: 52.524044
Test MAE: 108.73139
Test RMSE: 148.97289

FEDFUNDS
Validation MAE: 0.059834715
Validation RMSE: 0.15467909
Test MAE: 0.09463414
Test RMSE: 0.18572207

UNRATE
Validation MAE: 0.14049588
Validation RMSE: 0.188951
Test MAE: 0.2699115
Test RMSE: 1.0459784


## Neural forecasting models

The three models below are interchangeable: each takes a float tensor shaped `(batch, p, 1)` and returns `(batch, 1)`. Each model uses its own RevIN and a linear output head. Create a fresh model for each column and seed. The definitions are included in this notebook; no local package imports are needed in Colab.

### FCNN

Flatten the lag window, apply one ReLU hidden layer, and predict one value. The default architecture is **12 inputs -> 32 hidden units -> 1 output**, following the basic structure of Model 3 in the assigned paper.

In [11]:
class FCNN(_Forecaster):
    """Flattened lag window -> ReLU hidden layer(s) -> linear forecast.

    The default single hidden layer follows Model 3 of Almosova and Andresen.
    num_layers counts hidden layers, excluding the scalar output layer.
    """

    def __init__(self, lags: int = 12, hidden_dim: int = 32,
                 num_layers: int = 1, dropout: float = 0.0, eps: float = 1e-5):
        super().__init__(lags, hidden_dim, num_layers, dropout, eps)
        layers = [nn.Flatten(start_dim=1)]
        input_dim = lags
        for _ in range(num_layers):
            layers.extend([nn.Linear(input_dim, hidden_dim), nn.ReLU(),
                           nn.Dropout(dropout)])
            input_dim = hidden_dim
        layers.append(nn.Linear(hidden_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        normalized, stats = self._normalize(x)
        return self._restore(self.network(normalized), stats)

### LSTM

Use `torch.nn.LSTM` to read the window from oldest to newest. A linear head uses the final hidden state. Hidden state resets for each input window.

In [12]:
class LSTM(_Forecaster):
    """Unidirectional torch.nn.LSTM with a linear final-state prediction head."""

    def __init__(self, lags: int = 12, hidden_dim: int = 32,
                 num_layers: int = 1, dropout: float = 0.0, eps: float = 1e-5):
        super().__init__(lags, hidden_dim, num_layers, dropout, eps)
        self.lstm = nn.LSTM(
            input_size=1, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=False,
            # PyTorch's internal dropout only applies between stacked layers.
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def _encode(self, x: Tensor) -> tuple[Tensor, WindowStatistics]:
        normalized, stats = self._normalize(x)
        # No hidden state is passed in: each input window starts independently.
        hidden_states, _ = self.lstm(normalized)
        return hidden_states, stats

    def forward(self, x: Tensor) -> Tensor:
        hidden_states, stats = self._encode(x)
        return self._restore(self.head(hidden_states[:, -1, :]), stats)

### LSTM with attention

Use the final hidden state as query and every hidden state as both key and value, exactly as specified in the assignment:

$$q=h_p,\quad k_t=v_t=h_t,\quad s_t=\frac{q^\top k_t}{\sqrt{d}},\quad \alpha_t=\operatorname{softmax}(s)_t,\quad c=\sum_t\alpha_t h_t.$$

A linear head predicts from context vector $c$. `forward_with_attention` also returns the weights for interpretation: the first weight is lag 12 and the last is lag 1.

In [13]:
class ScaledDotProductAttention(nn.Module):
    """Use the last hidden state as query and all hidden states as keys/values."""

    def forward(self, hidden_states: Tensor) -> tuple[Tensor, Tensor]:
        # hidden_states: (batch, lags, hidden_dim)
        query = hidden_states[:, -1, :]
        scores = torch.bmm(hidden_states, query.unsqueeze(-1)).squeeze(-1)
        scores = scores / math.sqrt(hidden_states.shape[-1])
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)
        return context, weights


class LSTMWithAttention(LSTM):
    """Same LSTM backbone, with the homework's attention-weighted context head.

    forward(x) returns forecasts like the other models. For interpretation,
    forward_with_attention(x) also returns (batch, lags) attention weights.
    Weight index 0 is the oldest input (lag p); index -1 is the newest (lag 1).
    """

    def __init__(self, lags: int = 12, hidden_dim: int = 32,
                 num_layers: int = 1, dropout: float = 0.0, eps: float = 1e-5):
        super().__init__(lags, hidden_dim, num_layers, dropout, eps)
        self.attention = ScaledDotProductAttention()

    def forward_with_attention(self, x: Tensor) -> tuple[Tensor, Tensor]:
        hidden_states, stats = self._encode(x)
        context, weights = self.attention(hidden_states)
        return self._restore(self.head(context), stats), weights

    def forward(self, x: Tensor) -> Tensor:
        prediction, _ = self.forward_with_attention(x)
        return prediction

### Connect the models to the existing NumPy datasets

`prepare_datasets` returns inputs shaped `(N, p)` and targets shaped `(N,)`. These helpers add the dimensions PyTorch expects. `predict_model` converts predictions back to `(N,)` for the existing MAE/RMSE functions. Pass the raw prepared windows; RevIN is internal to each model.

In [14]:
MODEL_CLASSES = {"FCNN": FCNN, "LSTM": LSTM,
                 "LSTM with attention": LSTMWithAttention}


def create_model(model_name, seed=1):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    return MODEL_CLASSES[model_name](**MODEL_CONFIG)


def input_tensor(X):
    X = torch.as_tensor(X, dtype=torch.float32)
    if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] != p:
        raise ValueError(f"Expected nonempty inputs of shape (N, {p}).")
    if not torch.isfinite(X).all():
        raise ValueError("Inputs must be finite.")
    return X.unsqueeze(-1)


def target_tensor(y, n):
    y = torch.as_tensor(y, dtype=torch.float32).reshape(-1, 1)
    if y.shape[0] != n or not torch.isfinite(y).all():
        raise ValueError("Targets must be finite and match the input count.")
    return y


def predict_model(model, X, batch_size=256):
    device = next(model.parameters()).device
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in input_tensor(X).split(batch_size):
            predictions.append(model(batch.to(device)).cpu())
    return torch.cat(predictions).numpy().reshape(-1)

### Shared training function

All models use Adam at 0.001, batch size 32, supplied-scale MSE, at most 200 epochs, and patience 20. We restore the checkpoint with the lowest validation MSE. This function takes only training and validation arrays; test data cannot influence early stopping.

Shuffling is limited to complete windows within the training partition. The order inside each window and the chronological split are preserved.

In [15]:
def fit_model(model, X_train, y_train, X_val, y_val, *,
              learning_rate=0.001, batch_size=32, max_epochs=200,
              patience=20, seed=1, device="cpu"):
    if min(batch_size, max_epochs, patience) < 1 or learning_rate <= 0:
        raise ValueError("Training settings must be positive.")
    torch.manual_seed(seed)
    model.to(device)
    train_x = input_tensor(X_train)
    train_y = target_tensor(y_train, len(train_x))
    val_x = input_tensor(X_val).to(device)
    val_y = target_tensor(y_val, len(val_x)).to(device)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(train_x, train_y), batch_size=batch_size,
                        shuffle=True, generator=generator)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    history = []
    best_loss = float("inf")
    best_state = None
    best_epoch = 0
    stale_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            if not torch.isfinite(loss):
                raise RuntimeError("Training loss became non-finite.")
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_x)

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(val_x), val_y).item()
        if not math.isfinite(val_loss):
            raise RuntimeError("Validation loss became non-finite.")
        history.append({"epoch": epoch,
                        "train_mse": total_loss / len(train_x),
                        "val_mse": val_loss})
        if val_loss < best_loss:
            best_loss, best_epoch = val_loss, epoch
            best_state = {name: value.detach().cpu().clone()
                          for name, value in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    return {"history": pd.DataFrame(history), "best_epoch": best_epoch,
            "best_val_mse": best_loss}

### Verify integration with every selected column

This cell checks shapes, normalization, attention, and gradients using a small training batch from each real series. These are implementation checks with untrained models, not forecasting results.

In [16]:
for column_name in columns:
    X_train, y_train, *_ = prepare_datasets(extracted, column_name, p)
    batch_x = input_tensor(X_train[:4])
    batch_y = target_tensor(y_train[:4], len(batch_x))
    for model_name in MODEL_CLASSES:
        check_model = create_model(model_name)
        prediction = check_model(batch_x)
        assert prediction.shape == batch_y.shape
        assert torch.isfinite(prediction).all()
        nn.functional.mse_loss(prediction, batch_y).backward()
        assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all()
                   for parameter in check_model.parameters())
        normalized, stats = check_model.revin.normalize(batch_x)
        torch.testing.assert_close(check_model.revin.denormalize(normalized, stats), batch_x)
        if model_name == "LSTM with attention":
            _, weights = check_model.forward_with_attention(batch_x)
            torch.testing.assert_close(weights.sum(dim=1), torch.ones(len(batch_x)))
    print(f"{column_name}: all three models passed integration checks.")

S&P 500: all three models passed integration checks.
FEDFUNDS: all three models passed integration checks.
UNRATE: all three models passed integration checks.


### Fit and compare the final configurations

Set `RUN_EXPERIMENT = True` to train all three architectures for all three columns across seeds 1-5 (45 fits). It is off by default so running the notebook to review the implementation does not launch the full experiment.

The test set is evaluated only after selecting each run's checkpoint on validation data. Report mean and standard deviation of MAE/RMSE across seeds within each series. The baseline and neural models use the same retained test windows. A missing UNRATE observation excludes affected UNRATE windows; other columns retain their valid observations.

`trained_models[(column_name, model_name, seed)]` stores each fitted model for later forecast and attention plots. The earlier six-year rolling proposal is not implemented here.

In [17]:
RUN_EXPERIMENT = False

if RUN_EXPERIMENT:
    results = []
    trained_models = {}
    training_histories = {}
    test_forecasts = {}
    for column_name in columns:
        X_train, y_train, X_val, y_val, X_test, y_test = prepare_datasets(
            extracted, column_name, p)
        baseline = persistence_predict(X_test)
        # Repeat the deterministic baseline per seed for the same summary format.
        for seed in SEEDS:
            results.append({"column": column_name, "model": "Persistence",
                            "seed": seed, "mae": mae(y_test, baseline),
                            "rmse": rmse(y_test, baseline)})
        for model_name in MODEL_CLASSES:
            for seed in SEEDS:
                model = create_model(model_name, seed=seed)
                training = fit_model(model, X_train, y_train, X_val, y_val,
                                     seed=seed, **TRAINING_CONFIG)
                predictions = predict_model(model, X_test)
                key = (column_name, model_name, seed)
                trained_models[key] = model
                training_histories[key] = training
                test_forecasts[key] = predictions
                results.append({"column": column_name, "model": model_name,
                                "seed": seed, "mae": mae(y_test, predictions),
                                "rmse": rmse(y_test, predictions)})
                print(f"Finished {column_name} / {model_name} / seed {seed}; "
                      f"best validation epoch: {training['best_epoch']}")

    results = pd.DataFrame(results)
    summary = results.groupby(["column", "model"])[["mae", "rmse"]].agg(["mean", "std"])
    display(summary)
else:
    print("Models are ready. Set RUN_EXPERIMENT = True to run the full comparison.")

Models are ready. Set RUN_EXPERIMENT = True to run the full comparison.


### Inspect trained attention weights

After the experiment, this cell displays weights for three test examples. Column labels are lag numbers, from oldest to newest. The weights describe the model's allocation of attention, not causal effects.

In [18]:
if RUN_EXPERIMENT:
    attention_column = "UNRATE"  # Change to any selected column.
    attention_seed = SEEDS[0]
    attention_model = trained_models[(attention_column, "LSTM with attention", attention_seed)]
    *_, X_test, y_test = prepare_datasets(extracted, attention_column, p)
    example_indices = [0, len(X_test) // 2, len(X_test) - 1]
    attention_model.eval()
    with torch.no_grad():
        forecasts, attention_weights = attention_model.forward_with_attention(
            input_tensor(X_test[example_indices]).to(next(attention_model.parameters()).device))
    display(pd.DataFrame(attention_weights.cpu().numpy(),
                         index=example_indices,
                         columns=[f"lag_{lag}" for lag in range(p, 0, -1)]))

### Implementation references

- Local assignment: **HW1_Part2.pdf**, sections 3-5.
- [Almosova and Andresen, Model 3, section 2.1](https://www.ecb.europa.eu/press/conferences/shared/pdf/20190923_inflation_conference/L2_Almosova.pdf).
- [Kim et al. (2022), RevIN authors' implementation](https://github.com/ts-kim/RevIN).
- [Dive into Deep Learning: Attention Scoring Functions](https://d2l.ai/chapter_attention-mechanisms-and-transformers/attention-scoring-functions.html).
- [PyTorch LSTM documentation](https://docs.pytorch.org/docs/2.14/generated/torch.nn.LSTM.html).